In [49]:
# automatically reload imported modules before executing code

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
from pyrekordbox import Rekordbox6Database
import polars as pl
from nbutils import setup_path

setup_path()
db = Rekordbox6Database()

pl.Config.set_tbl_rows(20)  # Show 100 row

[13:13:17] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


polars.config.Config

In [78]:
from utils import get_base_dataset

base_df = get_base_dataset(db, min_tag_count=5)


Filtering tags with fewer than 5 occurrences:

Genre:
  - Grime: 1 occurrence(s)
  - New Beat: 3 occurrence(s)
  - Dancehall: 3 occurrence(s)
  - Blues: 4 occurrence(s)
  - Gabber: 4 occurrence(s)

Mood:
  - Industrial: 1 occurrence(s)

Total tags filtered: 6



In [79]:
from sklearn.model_selection import train_test_split

# replace with ALL songs 
features_df = pl.read_parquet("../data/song_features.parquet")

Xy_df = (
    base_df
    .join(features_df, on=["song_id", "song_path"], how="left")
    .filter(pl.col("has_tags"))
    .drop([
        "song_path", 
        "song_title", 
        "artist_id",
        "artist_name",
        "genre_id",
        "genre_name",
        "date_created",
        "tag_id",
        "sample_rate",
        "has_tags",
        "harmonic_percussive_ratio",
        "percussive_strength",
        "tonnetz_mean_0",
        "tonnetz_mean_1",	
        "tonnetz_mean_2",	
        "tonnetz_mean_3",	
        "tonnetz_mean_4",
        "tonnetz_mean_5",
        "tonnetz_std_0",
        "tonnetz_std_1",
        "tonnetz_std_2",
        "tonnetz_std_3",
        "tonnetz_std_4",	
        "tonnetz_std_5"
    ])
    .filter(
        pl.col("energy_increase_ratio").is_not_null()
    )
)


Xy_mood_df = (
    Xy_df
    .filter(pl.col("tag_group") == "Mood")
    .drop("tag_group")
)

Xy_situation_df = (
    Xy_df
    .filter(pl.col("tag_group") == "Situation")
    .drop("tag_group")
)

Xy_genre_df = (
    Xy_df
    .filter(pl.col("tag_group") == "Genre")
    .drop("tag_group")
)

y_label = "tag_name"
test_size = 0.2

# Split mood data
X_mood = Xy_mood_df.drop(y_label)
y_mood = Xy_mood_df.select(y_label)
X_mood_train, X_mood_test, y_mood_train, y_mood_test = train_test_split(
    X_mood, y_mood, test_size=test_size, random_state=42, stratify=Xy_mood_df[y_label]
)

# Split situation data
X_situation = Xy_situation_df.drop(y_label)
y_situation = Xy_situation_df.select(y_label)
X_situation_train, X_situation_test, y_situation_train, y_situation_test = train_test_split(
    X_situation, y_situation, test_size=test_size, random_state=42, stratify=Xy_situation_df[y_label]
)

# Split genre data
X_genre = Xy_genre_df.drop(y_label)
y_genre = Xy_genre_df.select(y_label)
X_genre_train, X_genre_test, y_genre_train, y_genre_test = train_test_split(
    X_genre, y_genre, test_size=test_size, random_state=42, stratify=Xy_genre_df[y_label]
)

In [80]:
X_mood_train

song_id,bpm,length,mel_spec_mean_0,mel_spec_mean_1,mel_spec_mean_2,mel_spec_mean_3,mel_spec_mean_4,mel_spec_mean_5,mel_spec_mean_6,mel_spec_mean_7,mel_spec_mean_8,mel_spec_mean_9,mel_spec_mean_10,mel_spec_mean_11,mel_spec_mean_12,mel_spec_mean_13,mel_spec_mean_14,mel_spec_mean_15,mel_spec_mean_16,mel_spec_mean_17,mel_spec_mean_18,mel_spec_mean_19,mel_spec_mean_20,mel_spec_mean_21,mel_spec_mean_22,mel_spec_mean_23,mel_spec_mean_24,mel_spec_mean_25,mel_spec_mean_26,mel_spec_mean_27,mel_spec_mean_28,mel_spec_mean_29,mel_spec_mean_30,mel_spec_mean_31,mel_spec_mean_32,mel_spec_mean_33,…,spectral_rolloff_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_contrast_mean_0,spectral_contrast_mean_1,spectral_contrast_mean_2,spectral_contrast_mean_3,spectral_contrast_mean_4,spectral_contrast_mean_5,spectral_contrast_mean_6,spectral_contrast_std_0,spectral_contrast_std_1,spectral_contrast_std_2,spectral_contrast_std_3,spectral_contrast_std_4,spectral_contrast_std_5,spectral_contrast_std_6,spectral_flatness_mean,spectral_flatness_std,tempo_0,beat_count,beat_regularity,zcr_mean,zcr_std,rms_mean,rms_std,rms_max,dynamic_range,rms_delta_mean,rms_delta_std,onset_strength_mean,onset_strength_std,onset_rate,energy_start,energy_middle,energy_end,energy_increase_ratio
str,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""65069301""",11100,371,-37.047718,-31.644592,-34.113922,-36.234352,-34.17276,-34.121979,-34.832932,-35.692497,-36.00724,-37.095268,-36.865597,-36.722527,-39.132877,-42.903885,-38.825844,-35.011833,-41.972218,-43.422722,-41.836327,-45.281052,-49.255619,-49.901703,-43.687408,-41.0112,-43.533745,-48.680767,-48.613789,-48.353821,-47.813061,-48.960571,-43.383801,-39.985806,-46.847633,-51.718311,…,2083.708884,3033.135381,583.594407,18.1597,10.166348,15.653031,16.374209,16.381225,17.601475,52.323711,5.536748,4.088167,4.01457,4.013214,3.090544,3.046965,3.538389,0.003377,0.026807,109.956782,672,140.690502,0.059542,0.056162,0.098126,0.081705,0.391738,0.391733,0.017274,0.033284,1.70493,1.567685,3.893536,0.138044,0.202641,0.151873,1.100167
"""204458084""",9540,213,-24.755754,-19.492464,-20.579065,-22.754868,-25.120344,-29.824383,-33.120239,-33.909336,-32.681885,-34.518349,-34.514507,-36.537407,-38.925491,-37.42329,-37.974751,-35.418049,-33.631664,-33.798439,-35.155979,-34.820297,-33.522953,-33.467003,-36.03442,-39.234924,-35.235474,-35.098789,-37.3866,-39.162941,-37.552761,-39.309723,-41.05851,-36.847843,-36.712986,-37.812447,…,2501.219296,4105.821653,615.818228,14.35756,10.061714,14.598744,16.548121,16.694818,16.27835,54.52427,5.116315,3.644053,4.073398,4.182729,3.4834,2.567451,2.948988,0.003947,0.011889,95.703125,324,95.57896,0.080319,0.087306,0.163791,0.07754,0.425215,0.425209,0.01908,0.028327,2.197682,1.354194,3.660372,0.266494,0.302025,0.240726,0.903302
"""238045823""",12630,353,-25.043629,-20.41107,-25.290728,-27.358238,-29.347466,-33.320457,-33.366211,-34.753036,-33.702236,-36.016193,-33.056767,-32.068844,-35.511585,-37.036507,-37.216362,-39.586151,-42.352852,-41.358814,-44.108559,-44.757648,-43.958622,-44.164455,-41.689213,-40.367905,-43.156044,-49.080555,-47.360909,-47.624348,-47.606361,-48.220787,-48.122898,-47.326595,-49.190453,-48.973869,…,3291.249235,3346.16983,906.154705,16.566021,11.552303,15.642066,16.092543,14.804282,16.001721,53.028093,4.944468,4.162562,4.607348,4.728809,3.403596,2.767663,5.473349,0.002919,0.037728,126.048018,723,53.508616,0.045857,0.044481,0.139622,0.095758,0.40966,0.40966,0.014813,0.027219,1.707923,1.379854,4.370965,0.194241,0.26323,0.233944,1.204393
"""71441805""",9750,198,-33.232159,-29.477764,-29.916252,-29.363092,-30.020794,-31.724476,-32.537643,-35.125549,-33.989475,-33.404591,-30.548903,-32.581051,-34.714657,-32.990032,-34.401623,-34.166794,-34.542175,

In [28]:
Xy_mood_df[[y_label, "song_id"]].group_by(y_label).count().filter(pl.col("count") < 5)[0:5]

/var/folders/2y/wcq7n_b14cqd36yz8rymh1540000gn/T/ipykernel_33433/3046250214.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  Xy_mood_df[[y_label, "song_id"]].group_by(y_label).count().filter(pl.col("count") < 5)[0:5]


agg_tag,count
str,u32
"""Dancefloor, Good Vibes, Groovy…",4
"""Dancefloor, Groovy""",3
"""Chill, Good Vibes, Pretty, Upl…",3
"""Dancefloor, Good Vibes, Groovy…",2
"""Chill, Dancefloor, Good Vibes,…",3


In [15]:
Xy_mood_df

song_id,bpm,length,agg_tag,mel_spec_mean_0,mel_spec_mean_1,mel_spec_mean_2,mel_spec_mean_3,mel_spec_mean_4,mel_spec_mean_5,mel_spec_mean_6,mel_spec_mean_7,mel_spec_mean_8,mel_spec_mean_9,mel_spec_mean_10,mel_spec_mean_11,mel_spec_mean_12,mel_spec_mean_13,mel_spec_mean_14,mel_spec_mean_15,mel_spec_mean_16,mel_spec_mean_17,mel_spec_mean_18,mel_spec_mean_19,mel_spec_mean_20,mel_spec_mean_21,mel_spec_mean_22,mel_spec_mean_23,mel_spec_mean_24,mel_spec_mean_25,mel_spec_mean_26,mel_spec_mean_27,mel_spec_mean_28,mel_spec_mean_29,mel_spec_mean_30,mel_spec_mean_31,mel_spec_mean_32,…,spectral_rolloff_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_contrast_mean_0,spectral_contrast_mean_1,spectral_contrast_mean_2,spectral_contrast_mean_3,spectral_contrast_mean_4,spectral_contrast_mean_5,spectral_contrast_mean_6,spectral_contrast_std_0,spectral_contrast_std_1,spectral_contrast_std_2,spectral_contrast_std_3,spectral_contrast_std_4,spectral_contrast_std_5,spectral_contrast_std_6,spectral_flatness_mean,spectral_flatness_std,tempo_0,beat_count,beat_regularity,zcr_mean,zcr_std,rms_mean,rms_std,rms_max,dynamic_range,rms_delta_mean,rms_delta_std,onset_strength_mean,onset_strength_std,onset_rate,energy_start,energy_middle,energy_end,energy_increase_ratio
str,i32,i32,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""100050502""",11940,260,"""Good Vibes, Sunset, Uplifting""",-43.239811,-31.118135,-25.137508,-24.333981,-25.647871,-29.55674,-31.613869,-36.069199,-36.379227,-38.59034,-39.525345,-43.366554,-43.600636,-39.561787,-41.165947,-44.384617,-46.44788,-47.45483,-48.650272,-46.117336,-43.572529,-43.846962,-46.95266,-50.178741,-45.715874,-46.023788,-44.056568,-41.502151,-41.577969,-44.293217,-46.385662,-44.068459,-46.924423,…,2101.983789,2897.985161,609.403001,20.636161,13.494756,15.636104,19.398142,16.690496,15.549656,52.527159,5.551661,4.939439,4.229432,5.152605,3.320459,2.261146,4.959294,0.008145,0.086644,120.18532,510,60.039977,0.039898,0.037762,0.060848,0.040342,0.25788,0.25788,0.010842,0.020448,1.552506,1.178102,7.658503,0.089394,0.110782,0.104654,1.170696
"""100643833""",12700,364,"""Dancefloor, Good Vibes, Groovy…",-29.006409,-19.751764,-18.16321,-22.282803,-26.009537,-27.191927,-27.654467,-30.44282,-30.357567,-34.518581,-37.271473,-37.915722,-38.648792,-37.158134,-37.174686,-39.61911,-42.59832,-40.615585,-44.60347,-44.488609,-41.374313,-42.614532,-40.986248,-40.074375,-42.524464,-45.79306,-43.884415,-44.700897,-44.80088,-45.287571,-44.690205,-42.971325,-45.44239,…,3042.173417,3572.188529,994.350352,19.34493,11.672536,14.845161,15.395565,15.838398,17.071376,52.176092,5.231313,4.547323,4.664573,4.491313,3.997642,3.376407,3.445935,0.001484,0.004487,126.048018,712,174.310095,0.03809,0.023762,0.104058,0.067284,0.408762,0.408758,0.020699,0.03332,1.994646,2.380835,5.953754,0.17499,0.177955,0.17005,0.971767
"""10108318""",12700,199,"""Dancefloor, Good Vibes, Sunset…",-36.109982,-26.503256,-26.449568,-28.075367,-26.882833,-26.908447,-27.64584,-27.461878,-26.229786,-26.482695,-27.57234,-31.27836,-28.69463,-29.29208,-35.76059,-31.731285,-27.67766,-29.719976,-39.873497,-36.916454,-37.429806,-44.787209,-45.212055,-47.283291,-44.058556,-42.557987,-41.643265,-45.59618,-49.285995,-46.118645,-45.24287,-47.79398,-45.814625,…,3309.806276,3383.746338,998.057007,15.41582,10.721512,18.260421,19.16219,19.492784,19.545071,56.806785,5.183443,3.724683,4.829542,5.429297,5.591766,5.135531,5.813163,0.009032,0.093753,126.048018,354,98.082896,0.062027,0.044844,0.032591,0.020408,0.108729,0.108729,0.003847,0.006857,1.616246,1.729718,3.70098,0.054717,0.061379,0.045351,0.828822
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"

In [12]:
Xy_genre_df[0:5]


song_id,bpm,length,agg_tag,mel_spec_mean_0,mel_spec_mean_1,mel_spec_mean_2,mel_spec_mean_3,mel_spec_mean_4,mel_spec_mean_5,mel_spec_mean_6,mel_spec_mean_7,mel_spec_mean_8,mel_spec_mean_9,mel_spec_mean_10,mel_spec_mean_11,mel_spec_mean_12,mel_spec_mean_13,mel_spec_mean_14,mel_spec_mean_15,mel_spec_mean_16,mel_spec_mean_17,mel_spec_mean_18,mel_spec_mean_19,mel_spec_mean_20,mel_spec_mean_21,mel_spec_mean_22,mel_spec_mean_23,mel_spec_mean_24,mel_spec_mean_25,mel_spec_mean_26,mel_spec_mean_27,mel_spec_mean_28,mel_spec_mean_29,mel_spec_mean_30,mel_spec_mean_31,mel_spec_mean_32,…,spectral_contrast_std_4,spectral_contrast_std_5,spectral_contrast_std_6,spectral_flatness_mean,spectral_flatness_std,tempo_0,beat_count,beat_regularity,zcr_mean,zcr_std,rms_mean,rms_std,rms_max,dynamic_range,rms_delta_mean,rms_delta_std,onset_strength_mean,onset_strength_std,onset_rate,harmonic_percussive_ratio,percussive_strength,tonnetz_mean_0,tonnetz_mean_1,tonnetz_mean_2,tonnetz_mean_3,tonnetz_mean_4,tonnetz_mean_5,tonnetz_std_0,tonnetz_std_1,tonnetz_std_2,tonnetz_std_3,tonnetz_std_4,tonnetz_std_5,energy_start,energy_middle,energy_end,energy_increase_ratio
str,i32,i32,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""101186312""",12500,307,"""Disco, House, Soul""",-29.423384,-23.397461,-24.85667,-26.253923,-27.035971,-29.686871,-31.475702,-33.475342,-32.732586,-35.248451,-33.868217,-33.554482,-32.681599,-32.031067,-34.792839,-35.570915,-35.43681,-34.77602,-36.818577,-34.231667,-33.948208,-34.847866,-33.12986,-35.462841,-32.58445,-31.560507,-30.093067,-31.983606,-34.948769,-35.903339,-36.314465,-36.404659,-37.030231,…,3.645502,2.388977,3.014435,0.000371,0.008688,126.048018,642,82.926945,0.111607,0.059316,0.123436,0.079819,0.366275,0.366275,0.01659,0.027356,1.799584,1.090185,5.363796,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125862,0.225122,0.261071,2.074249
"""101533884""",11540,248,"""Disco, Funk, Motown, Soul""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""104519868""",12030,220,"""Disco, Funk, Motown, Soul""",-29.345961,-22.166965,-22.611174,-23.48035,-24.391569,-24.441399,-24.565968,-28.946365,-30.587833,-31.864643,-32.067699,-33.466675,-32.629662,-33.228188,-35.011116,-33.945976,-36.897312,-37.949047,-37.326763,-34.622059,-36.19384,-36.588047,-35.720211,-39.215324,-35.419807,-35.647224,-35.899658,-38.266819,-37.230213,-33.986931,-35.281334,-38.766209,-39.743168,…,3.558816,3.666878,4.936307,0.008619,0.086499,120.18532,420,40.958224,0.070718,0.028739,0.075639,0.042733,0.2555,0.2555,0.010312,0.018779,1.85072,1.341241,4.240784,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.127579,0.14171,0.106869,0.837662
"""104573928""",13000,284,"""Pop""",-32.276688,-24.811747,-24.341211,-26.628817,-28.909037,-30.476707,-32.24781,-37.240314,-34.164787,-35.037178,-35.195568,-35.975254,-38.722103,-40.79232,-41.223289,-40.810902,-42.393669,-39.364754,-40.377247,-41.378242,-41.257462,-42.110851,-41.64222,-42.19471,-42.348515,-44.771992,-42.588299,-40.423313,-40.99258,-44.311325,-44.431343,-42.663002,-45.688934,…,3.788285,3.902244,3.397859,0.002196,0.023996,129.199219,521,122.62524,0.051651,0.044768,0.111878,0.076536,0.425036,0.425036,0.016154,0.027457,1.772913,1.194108,3.129249,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.16467,0.226503,0.165084,1.002503
"""104652117""",10590,337,"""R&B, Soul""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,n